In [ ]:
!pip install transformers # Installing the transformers library

     |████████████████████████████████| 573kB 2.7MB/s 
     |████████████████████████████████| 1.0MB 11.8MB/s 
     |████████████████████████████████| 890kB 19.7MB/s 
     |████████████████████████████████| 3.7MB 23.2MB/s 
  Created wheel for sacremoses: filename=sacremoses-0.0.41-cp36-none-any.whl size=893334 sha256=0b2edaffac056e0e70f508ab0538163d9c6962e8d3aa7a58b1edcbfe91de092a
  Stored in directory: /root/.cache/pip/wheels/22/5a/d4/b020a81249de7dc63758a34222feaa668dbe8ebfe9170cc9b1
Successfully built sacremoses


In [ ]:
import transformers # transformers library
import torch # PyTorch, we are using PyTorch as our library

In [ ]:
# We are going to load in GPT-2 using the transformers library
gpt_tokenizer = transformers.GPT2Tokenizer.from_pretrained('gpt2-large')
# Loading in model now...
gpt_model = transformers.GPT2LMHeadModel.from_pretrained('gpt2-large')
# Takes a while to run...

In [ ]:
## Making a function that will generate text for us ##
def gen_text(prompt_text, tokenizer, model, n_seqs=1, max_length=25):
  # n_seqs is the number of sentences to generate
  # max_length is the maximum length of the sentence
  encoded_prompt = tokenizer.encode(prompt_text, add_special_tokens=False, return_tensors="pt")
  # We are encoding the text using the gpt tokenizer. The return tensors are of type "pt"
  # since we are using PyTorch, not tensorflow
  output_sequences = model.generate(
      input_ids=encoded_prompt,
      max_length=max_length+len(encoded_prompt), # The model has to generate something,
      # so we add the length of the original sequence to max_length
      temperature=1.0,
      top_k=0,
      top_p=0.9,
      repetition_penalty=1.2, # To ensure that we dont get repeated phrases
      do_sample=True,
      num_return_sequences=n_seqs
  ) # We feed the encoded input into the model.
  ## Getting the output ##
  if len(output_sequences.shape) > 2:
    output_sequences.squeeze_() # the _ indicates that the operation will be done in-place
  generated_sequences = []
  for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
    generated_sequence = generated_sequence.tolist()
    text = tokenizer.decode(generated_sequence)
    total_sequence = (
        prompt_text + text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True, )) :]
    )
    generated_sequences.append(total_sequence)
  return generated_sequences

In [ ]:
# Lots of syntax errors, but now we can test our model
## One important note: in our function, on line 5, make sure that
# return_tensor is return_tensors, otherwise you will get an error like
# this:
#####
# Another important note: on line 27 of the function, instead of
# clear_up_tokenization_spaces, write clean_up_tokenization_spaces
####
gen_text("Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry",gpt_tokenizer,gpt_model)

Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


['Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry. But once the two companies were in']

In [ ]:
# Sequence length was too small, lets increase it
gen_text("Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry",
         gpt_tokenizer,
         gpt_model,
         max_length=100)
# Will take some time......

Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


['Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry. Their front line seemed to be joined by dwarves from Watarland as they opened fire, but they were denied their usual axes or bows. The goblins opened up in the rear, plucking at the embers of burning logs. With great effort and gnashing teeth the Beornings struck back at them, shouting "No more!" to warn those near that they would go down fighting no matter what.\n\n']

In [ ]:
# We can demostrate n_seqs here
gen_text("Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry",
         gpt_tokenizer,
         gpt_model,
         max_length=40,
         n_seqs=3) # Will take even longer....

Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


['Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry. All that remained was Thorin\'s staff as he stood in awe.\n\n"Speak of me yet',
 'Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry. With both eyes bleeding, Tauriel did her best to give them what they wanted—but she was soon',
 "Legolas and Gimli advanced on the orcs, raising their weapons with a harrowing war cry. The desperate First Men stood motionless as Beren's arrival scared off the fresh attacking army. They threw down"]

In [ ]:
# There are now 3 different outputs
# thats it